In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator

from statsmodels.tsa.seasonal import seasonal_decompose

import helper

# Section b: DataAssignment1

First: for each method, consider
- estimation set t = 1,...,30
- forecast set t = 31,...,40
- tune (when needed) using t = 31,...,40
- forecast precision 

Based on all nine time variables, make a recommendation. 

In [2]:
df_1 = pd.read_excel("DataAssignment1.xlsx")

N = 50
y_list = [df_1.iloc[:, i].astype(float).to_numpy() for i in range(9)]
y_names = [f"y_{i+1}" for i in range(9)]

t_est_end = 30        
t_fc_start = 30      
t_fc_end = 40

In [3]:
#evaluation function
def metrics(y_true, y_hat):
    u = y_true - y_hat
    return {
        "ME": np.mean(u),
        "MAE": np.mean(np.abs(u)),
        "MAPE": np.mean(100 * np.abs(u) / np.abs(y_true)),
        "MSE": np.mean(u**2),
    }

all_results = {} #store in dictionary 
exp_alpha_store = {} #store parameters for part 2
holt_param_store = {} #store parameters for part 2

In [4]:
#loop over all time series
for idx, y in zip(y_names, y_list):
    y_est = y[:t_est_end]
    y_eval = y[t_fc_start:t_fc_end]

    forecasts = {} #store in dictionary

    #average
    y_avg = np.mean(y_est)
    f_avg = np.full_like(y_eval, y_avg) #forecast = average
    forecasts["Average"] = f_avg

    #running average
    f_avg = helper.running_average_forecast(y)
    forecasts["Running Average"] = f_avg[t_fc_start:t_fc_end]

    #random walk
    f_rw = helper.lag_forecast(y)
    forecasts['Random Walk'] = f_rw[t_fc_start:t_fc_end]
    
    #random walk with drift
    _, rw_drift, _ = helper.random_walk_plus_drift_forecast(y)
    forecasts['Random Walk + Drift'] = rw_drift[t_fc_start:t_fc_end]

    #exponential smoothing (tuned based on mse)
    alpha_candidates = np.linspace(0.01, 0.99, 99) #(0,1)
    best_alpha, best_mse = None, np.inf
    for alpha in alpha_candidates:
        f_es = helper.exp_smoothing_forecast(y, alpha)
        f_eval = f_es[t_fc_start:t_fc_end]
        mse = np.mean((y_eval - f_eval)**2)
        if mse < best_mse:
            best_alpha, best_mse = alpha, mse
    
    f_es = helper.exp_smoothing_forecast(y, best_alpha)[t_fc_start:t_fc_end]
    forecasts[f'Exponentional Smoothing (alpha = {best_alpha:.3f})'] = f_es

    exp_alpha_store[idx] = best_alpha

    #holt winters/double exponential smoothing (tuned)
    alpha_path, beta_path, F_hat, u_hat, loss_path = helper.estimate_alpha_beta_holt_winters(
        y_est,
        criterion="SSE",
        grid_n=201,
        eps=1e-3,
    ) #like in a

    alpha = alpha_path[~np.isnan(alpha_path)][-1]
    beta = beta_path[~np.isnan(beta_path)][-1]
    f_holt = helper.holt_fitted_forecast_series(y, alpha, beta)
    f_holt[:2] = np.nan
    forecasts['Holt Winters'] = f_holt[t_fc_start: t_fc_end]

    holt_param_store[idx] = (alpha, beta)

    #evaluation
    res = {}
    for method, forecast in forecasts.items():
        res[method] = metrics(y_eval, forecast)

    all_results[idx] = res

In [5]:
#results

def normalize_method(name):
    name = name.lower()
    if "smoothing" in name or "expsmo" in name:
        return "ExpSmo (tuned)"
    elif "rw + drift" in name or "drift" in name:
        return "RW + Drift"
    elif "random walk" in name:
        return "Random Walk"
    elif "running" in name:
        return "Running Avg"
    elif "average" in name:
        return "Average"
    elif "holt" in name:
        return "Holt-Winters"
    else:
        return name
    
panels = {}

for var in all_results:
    df_tmp = pd.DataFrame(all_results[var]).T
    df_tmp.index = [normalize_method(m) for m in df_tmp.index]
    panels[var] = df_tmp

all_methods = sorted(
    set().union(*[df.index.tolist() for df in panels.values()])
)
agg = {m: {"ME": [], "MAE": [], "MAPE": [], "MSE": []} for m in all_methods}

for var, df in panels.items():
    for m in all_methods:
        if m in df.index:
            for k in ["ME", "MAE", "MAPE", "MSE"]:
                agg[m][k].append(df.loc[m, k])

for var, df_res in panels.items():
    print(f"\nFORECAST RESULTS — {var}  (t = 31…40)")
    print("="*70)
    print(df_res.round(3).to_string())
    print("="*70)


FORECAST RESULTS — y_1  (t = 31…40)
                   ME    MAE   MAPE     MSE
Average        -0.030  2.478  5.686   7.788
Running Avg    -0.229  2.598  5.987   8.023
Random Walk    -0.530  1.330  3.143   3.137
RW + Drift     -0.524  1.352  3.190   3.201
ExpSmo (tuned) -0.661  1.245  2.961   3.047
Holt-Winters   -1.121  2.928  6.800  10.643

FORECAST RESULTS — y_2  (t = 31…40)
                   ME    MAE    MAPE     MSE
Average         6.927  6.927  12.114  54.220
Running Avg     5.886  5.886  10.228  43.539
Random Walk    -0.670  1.690   3.036   4.701
RW + Drift     -1.233  1.924   3.464   5.862
ExpSmo (tuned) -0.891  1.707   3.071   4.056
Holt-Winters   -1.242  1.924   3.460   5.082

FORECAST RESULTS — y_3  (t = 31…40)
                   ME    MAE    MAPE     MSE
Average        -3.610  3.738  10.315  21.450
Running Avg    -2.953  3.454   9.532  18.768
Random Walk     0.910  2.050   5.276   6.403
RW + Drift      1.115  2.162   5.564   7.023
ExpSmo (tuned)  1.222  1.879   4.812   5.

In [6]:
#average results across variables
agg_df = pd.DataFrame({
    m: {k: np.mean(agg[m][k]) for k in agg[m] if len(agg[m][k]) > 0}
    for m in agg
}).T

print("\nAGGREGATED FORECAST PERFORMANCE (MEAN OVER y₁…y₉)")
print("=" * 80)
print(agg_df.round(4).to_string())
print("=" * 80)


AGGREGATED FORECAST PERFORMANCE (MEAN OVER y₁…y₉)
                    ME     MAE     MAPE      MSE
Average        -1.9270  5.8934  17.9616  53.5420
ExpSmo (tuned) -0.1651  1.4836   4.2325   3.4831
Holt-Winters   -0.1586  1.7815   4.9604   4.9966
RW + Drift     -0.0730  1.6736   4.7296   4.4315
Random Walk    -0.1211  1.6367   4.6769   4.2190
Running Avg    -1.7435  5.3121  16.1769  43.2808


Recommendation rule: choose model with lowest MAE / MSE / MAPE jointly. Based on the above aggregated forecsat performances presented, the tuned Exponential Smoothing model satisfied this criterion as the MAE, MSE, and MAPE are the lowest across all models. Hence, the tuned Exponential Smoothing model is the recommended choice.

Second:
- t = 41,...,50
- re-assess the forecast precisions
- forecast precision

Based on all nine time variables, make a recommendation. Is there a difference?

In [7]:
t_fc2_start = 40  
t_fc2_end   = 50 
all_results_2 = {}

In [8]:
for idx, y in zip(y_names, y_list):
    y_eval = y[t_fc2_start:t_fc2_end]
    forecasts = {}

    #average
    y_avg = np.mean(y[:t_est_end])
    forecasts["Average"] = np.full_like(y_eval, y_avg)

    #running average
    f_avg = helper.running_average_forecast(y)
    forecasts["Running Average"] = f_avg[t_fc2_start:t_fc2_end]

    #random walk
    f_rw = helper.lag_forecast(y)
    forecasts["Random Walk"] = f_rw[t_fc2_start:t_fc2_end]

    #random walk with drift
    _, rw_drift, _ = helper.random_walk_plus_drift_forecast(y)
    forecasts["Random Walk + Drift"] = rw_drift[t_fc2_start:t_fc2_end]

    #exponential smoothing (fixed alpha)
    alpha = exp_alpha_store[idx]
    f_es = helper.exp_smoothing_forecast(y, alpha)
    forecasts[f'Exponentional Smoothing (alpha = {alpha:.3f})'] = \
        f_es[t_fc2_start:t_fc2_end]

    #holt winter (fixed parameters)
    alpha, beta = holt_param_store[idx]
    f_holt = helper.holt_fitted_forecast_series(y, alpha, beta)
    f_holt[:2] = np.nan
    forecasts["Holt Winters"] = f_holt[t_fc2_start:t_fc2_end]

    #evaluation
    res = {}
    for method, forecast in forecasts.items():
        res[method] = metrics(y_eval, forecast)

    all_results_2[idx] = res

In [9]:
#results 
panels_2 = {}

for var in all_results_2:
    df_tmp = pd.DataFrame(all_results_2[var]).T
    df_tmp.index = [normalize_method(m) for m in df_tmp.index]
    panels_2[var] = df_tmp

for var, df_res in panels_2.items():
    print(f"\nFORECAST RESULTS — {var}  (t = 41…50)")
    print("="*70)
    print(df_res.round(3).to_string())
    print("="*70)



FORECAST RESULTS — y_1  (t = 41…50)
                   ME    MAE   MAPE    MSE
Average        -2.800  2.800  7.090  9.498
Running Avg    -2.491  2.491  6.324  8.079
Random Walk     0.150  1.410  3.538  2.971
RW + Drift      0.239  1.466  3.676  3.071
ExpSmo (tuned)  0.158  1.283  3.218  2.479
Holt-Winters    1.322  1.336  3.266  3.621

FORECAST RESULTS — y_2  (t = 41…50)
                   ME    MAE   MAPE     MSE
Average         0.807  3.700  7.279  16.779
Running Avg    -1.083  3.840  7.826  17.183
Random Walk    -0.840  1.660  3.373   3.612
RW + Drift     -1.136  1.851  3.752   4.158
ExpSmo (tuned) -1.264  1.895  3.879   4.874
Holt-Winters   -0.979  1.752  3.560   3.979

FORECAST RESULTS — y_3  (t = 41…50)
                   ME    MAE   MAPE    MSE
Average        -1.380  1.414  3.560  3.383
Running Avg    -0.457  1.057  2.646  1.698
Random Walk    -0.330  1.450  3.591  3.235
RW + Drift     -0.258  1.450  3.586  3.263
ExpSmo (tuned) -0.348  1.296  3.223  2.279
Holt-Winters   -0.712 

In [10]:
#average results
all_methods_2 = sorted(
    set().union(*[df.index.tolist() for df in panels_2.values()])
)

agg_2 = {m: {"ME": [], "MAE": [], "MAPE": [], "MSE": []} for m in all_methods_2}

for var, df in panels_2.items():
    for m in all_methods_2:
        if m in df.index:
            for k in ["ME", "MAE", "MAPE", "MSE"]:
                agg_2[m][k].append(df.loc[m, k])

agg_df_2 = pd.DataFrame({
    m: {k: np.mean(agg_2[m][k]) for k in agg_2[m] if len(agg_2[m][k]) > 0}
    for m in agg_2
}).T

print("\nAGGREGATED FORECAST PERFORMANCE (MEAN OVER y₁…y₉) — PART 2")
print("=" * 80)
print(agg_df_2.round(4).to_string())
print("=" * 80)



AGGREGATED FORECAST PERFORMANCE (MEAN OVER y₁…y₉) — PART 2
                    ME     MAE     MAPE      MSE
Average        -2.2337  5.6300  17.5362  51.7748
ExpSmo (tuned)  0.2203  1.5237   4.2650   3.7068
Holt-Winters    0.3247  1.5668   4.3996   4.0172
RW + Drift      0.1358  1.6241   4.6263   3.8657
Random Walk     0.0856  1.5433   4.3970   3.5926
Running Avg    -1.5696  4.1273  12.7750  29.0733


Based on the above aggregated forecsat performances presented, the tuned Exponential Smoothing model is no longer the model with lowest MSE and thus the recommendation must change. In the new results, there is no inferior model that minimizes a majority of the metric. The Random Walk actually has the lowest MSE, while Holt-Winters has the lowest MAE. The Random Walk is also a close runner-up in MAE and MAPE. This tells us that its simplicity overtakes the complex methods and perhaps avoids overfitting. Hence, the Random Walk model should now to be recommended. 